In [74]:
import torch

torch.set_printoptions(precision=2, sci_mode=False)

In [75]:
torch.manual_seed(233)
inputs = torch.randn(5, 3)
inputs # I am a good boy

tensor([[ 0.59,  0.50, -0.66],
        [ 1.33, -2.49, -0.96],
        [-1.04,  0.49,  0.22],
        [ 0.69,  0.42, -0.86],
        [ 0.16,  0.61, -1.64]])

In [76]:
torch.manual_seed(31)
d_in, d_out = 3, 2
W_Q = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
print("W_Q", W_Q.shape, W_Q)
W_K = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
print("W_K", W_K.shape, W_K)
W_V = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
print("W_V", W_V.shape, W_V)

W_Q torch.Size([3, 2]) Parameter containing:
tensor([[ 0.35, -0.57],
        [-0.39, -0.29],
        [-0.45,  0.92]])
W_K torch.Size([3, 2]) Parameter containing:
tensor([[-0.76,  1.06],
        [-0.12,  0.58],
        [ 0.47,  0.30]])
W_V torch.Size([3, 2]) Parameter containing:
tensor([[ 0.56,  2.29],
        [ 2.11,  0.30],
        [ 0.03, -0.57]])


In [77]:
# we treat good (vector 4) as query vector
query_vector = inputs[3]
print(f"{query_vector=}")
query_q = query_vector @ W_Q
query_k = query_vector @ W_K
query_v = query_vector @ W_V
print(query_q.shape, query_q)
print(query_k.shape, query_k)
print(query_v.shape, query_v)

query_vector=tensor([ 0.69,  0.42, -0.86])
torch.Size([2]) tensor([ 0.47, -1.30])
torch.Size([2]) tensor([-0.98,  0.71])
torch.Size([2]) tensor([1.24, 2.20])


In [78]:
# let's get inputs K and V
inputs_k = inputs @ W_K
inputs_v = inputs @ W_V
inputs_k, inputs_v

(tensor([[-0.81,  0.71],
         [-1.17, -0.32],
         [ 0.84, -0.75],
         [-0.98,  0.71],
         [-0.95,  0.03]]),
 tensor([[ 1.37,  1.87],
         [-4.56,  2.84],
         [ 0.47, -2.36],
         [ 1.24,  2.20],
         [ 1.32,  1.48]]))

In [79]:
# Now we will get the attention score
d_k = inputs_k.shape[-1]
qkt = query_q @ inputs_k.T
print(qkt)
qkt_norm = torch.softmax(qkt / d_k**0.5, dim=0)
print(qkt_norm)

tensor([-1.31, -0.13,  1.37, -1.39, -0.49])
tensor([0.08, 0.18, 0.52, 0.07, 0.14])


In [80]:
# now multiple the scores with the values 
qkt_norm @ inputs_v

tensor([-0.19, -0.21])

In [81]:
torch.manual_seed(31)
# A custom Pytorch class
class SelfAttenV1(torch.nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_Q = torch.nn.Parameter(torch.randn(d_in, d_out))
        self.W_K = torch.nn.Parameter(torch.randn(d_in, d_out))
        self.W_V = torch.nn.Parameter(torch.randn(d_in, d_out))

    def forward(self, x):
        x_k = x @ self.W_K
        x_v = x @ self.W_V
        x_q = x @ self.W_Q
        xqkt = x_q @ x_k.T
        xqkt_norm = torch.softmax((xqkt / (x_k.shape[-1] ** 0.5)), dim=-1)
        atten_weights = xqkt_norm @ x_v
        return atten_weights

In [82]:
self_attn = SelfAttenV1(3, 2)

In [ ]:
self_attn(
    inputs
)  # you can see the "good"'s context vector [-0.19, -0.21] is same like we did above

tensor([[-0.25,  0.18],
        [ 0.34, -1.66],
        [ 0.13,  1.81],
        [-0.19, -0.21],
        [-0.17, -0.63]], grad_fn=<MmBackward0>)

In [87]:
# self attention with nn.Linear (which leads to the stable training and weight init)
class SelfAttenV2(torch.nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_Q = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_K = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_V = torch.nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_Q(x)
        keys = self.W_K(x)
        values = self.W_V(x)
        print(queries.shape, keys.shape, values.shape)
        atten_scores = queries @ keys.T
        atten_scores_norm = torch.softmax(
            atten_scores / keys.shape[-1] ** 0.5,
            dim=-1,
        )
        atten_weights = atten_scores_norm @ values
        print(atten_weights.shape)
        return atten_weights

In [88]:
torch.manual_seed(31)
self_attn_v2 = SelfAttenV2(3, 2)
self_attn_v2

SelfAttenV2(
  (W_Q): Linear(in_features=3, out_features=2, bias=False)
  (W_K): Linear(in_features=3, out_features=2, bias=False)
  (W_V): Linear(in_features=3, out_features=2, bias=False)
)

In [90]:
print(inputs.shape)
self_attn_v2(inputs)

torch.Size([5, 3])
torch.Size([5, 2]) torch.Size([5, 2]) torch.Size([5, 2])
torch.Size([5, 2])


tensor([[-0.04,  0.38],
        [ 0.27,  0.57],
        [-0.03,  0.41],
        [-0.03,  0.39],
        [-0.04,  0.40]], grad_fn=<MmBackward0>)

In [91]:
self_attn_v2, self_attn

(SelfAttenV2(
   (W_Q): Linear(in_features=3, out_features=2, bias=False)
   (W_K): Linear(in_features=3, out_features=2, bias=False)
   (W_V): Linear(in_features=3, out_features=2, bias=False)
 ),
 SelfAttenV1())

In [101]:
# Exercise make atten and attenv2 same (same weights)
self_attn.W_Q = torch.nn.Parameter(self_attn_v2.W_Q.weight.T)
self_attn.W_K = torch.nn.Parameter(self_attn_v2.W_K.weight.T)
self_attn.W_V = torch.nn.Parameter(self_attn_v2.W_V.weight.T)

In [102]:
self_attn_v2(inputs)

torch.Size([5, 2]) torch.Size([5, 2]) torch.Size([5, 2])
torch.Size([5, 2])


tensor([[-0.04,  0.38],
        [ 0.27,  0.57],
        [-0.03,  0.41],
        [-0.03,  0.39],
        [-0.04,  0.40]], grad_fn=<MmBackward0>)

In [103]:
self_attn(inputs)

tensor([[-0.04,  0.38],
        [ 0.27,  0.57],
        [-0.03,  0.41],
        [-0.03,  0.39],
        [-0.04,  0.40]], grad_fn=<MmBackward0>)